In [ ]:
# ==============================================================================
# Cell 1: Kiểm tra phần cứng Dual-T4 & Thiết lập môi trường JAX
# ==============================================================================
import os, sys

os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.85"
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"

print("[+] Đang kiểm tra GPU qua nvidia-smi:")
!nvidia-smi --query-gpu=index,name,memory.total --format=csv,noheader

import jax
devices = jax.devices()
print(f"[✓] JAX thiết lập thành công! Số lượng thiết bị active: {len(devices)}")
for i, d in enumerate(devices):
    print(f"    GPU {i}: {d.device_kind} (platform={d.platform})")

In [ ]:
# ==============================================================================
# Cell 2: Chuẩn hóa đường dẫn dự án & Cài đặt thư viện simulation
# ==============================================================================
import glob, os, shutil, zipfile, subprocess

print("Thư mục hiện tại:", os.getcwd())
dst = "/kaggle/working/Bipedal-Locomotion"

zips = glob.glob("/kaggle/input/**/*.zip", recursive=True) + glob.glob("/kaggle/working/*.zip")
input_dirs = [d for d in glob.glob("/kaggle/input/*") if os.path.isdir(d)]

if zips and not os.path.exists(dst):
    print("Đang giải nén:", zips[0])
    os.makedirs(dst, exist_ok=True)
    with zipfile.ZipFile(zips[0], "r") as zf:
        zf.extractall(dst)
elif input_dirs and not os.path.exists(dst):
    print("Đang copy từ dataset input:", input_dirs[0])
    shutil.copytree(input_dirs[0], dst)

project_root = None
for base in [dst, "/kaggle/working", "/kaggle/input"]:
    for r, _, files in os.walk(base):
        if "setup_kaggle.sh" in files and "configs" in os.listdir(r):
            project_root = r
            break
    if project_root:
        break

if project_root and project_root.startswith("/kaggle/input"):
    if not os.path.exists(dst):
        shutil.copytree(project_root, dst)
    project_root = dst

if project_root:
    os.chdir(project_root)

print("Đã chuyển vào thư mục project:", os.getcwd())

# Cài đặt MuJoCo Playground & Brax phiên bản chuẩn tương thích
tag = "v0.2.0" if sys.version_info >= (3, 11) else "v0.1.0"
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--upgrade", "--no-deps",
    f"git+https://github.com/google-deepmind/mujoco_playground.git@{tag}"
])
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--upgrade",
    "brax", "flax", "optax", "orbax-checkpoint", "ml_collections", "etils", "lxml", "mujoco-mjx", "imageio", "imageio-ffmpeg"
])
print("[✓] Cài đặt dependencies hoàn tất!")

In [ ]:
# ==============================================================================
# Cell 3: Huấn luyện chuyên sâu cho Robot đi mượt trên Mặt phẳng (Flat Terrain)
# ==============================================================================
# Chú ý: Giữ nguyên file configs/terrain.yaml, nhưng tinh chỉnh cấu hình PPO
# giúp robot học đi ổn định, chống overfit và tăng tỷ lệ thành công.

import sys, subprocess

print("==========================================================")
print(" BẮT ĐẦU HUẤN LUYỆN PPO: UNITREE G1 TRÊN MẶT PHẲNG (FLAT)")
print("==========================================================")

# Sử dụng cấu hình larger_mlp.yaml (Model 2 capacity-matched) với task_name = G1JoystickFlatTerrain
cmd = [
    sys.executable, "scripts/train_m2.py",
    "--config", "configs/larger_mlp.yaml",
    "--seed", "42",
    "--num-timesteps", "50000000",
    "--output-dir", "results/flat_optimized"
]

subprocess.check_call(cmd)
print("[✓] Huấn luyện hoàn tất! Checkpoint lưu tại results/flat_optimized")

In [ ]:
# ==============================================================================
# Cell 4: Đánh giá định lượng hiệu năng đi bộ trên mặt phẳng
# ==============================================================================
import glob, os, sys, subprocess, json
import pandas as pd

run_dirs = sorted(glob.glob("results/flat_optimized/g1_m2_larger_mlp_*"))
assert run_dirs, "Chưa tìm thấy thư mục kết quả huấn luyện!"
latest_run = run_dirs[-1]
ckpt_dir = os.path.join(latest_run, "checkpoints")

out_json = os.path.join(latest_run, "eval_flat_nominal.json")
cmd = [
    sys.executable, "scripts/eval_m2.py",
    "configs/larger_mlp.yaml",
    "--checkpoint", ckpt_dir,
    "--scenario", "nominal",
    "--num-episodes", "20",
    "--output-json", out_json
]

subprocess.check_call(cmd)

with open(out_json) as f:
    res = json.load(f)

print("\n================ KẾT QUẢ ĐÁNH GIÁ ĐI BỘ TRÊN MẶT PHẲNG ================")
for k in ["scenario", "model_id", "mean_return", "mean_length", "success_rate", "fall_rate", "mean_vel_error"]:
    print(f"  {k:<20}: {res.get(k)}")
print("======================================================================\n")

In [ ]:
# ==============================================================================
# Cell 5: Render Video MP4 Demo Robot bước đi mượt mà & Hiển thị trực tiếp
# ==============================================================================
import os, glob, sys
from pathlib import Path
import numpy as np
from IPython.display import HTML
from base64 import b64encode

os.environ["MUJOCO_GL"] = "egl"

import jax
import mujoco
from mujoco import mjx
import imageio

from src.training.config_loader import load_training_config
from src.training.env_builder import build_g1_environment
from src.models.mlp_policy import build_model_spec_from_config, make_model_network_factory
from src.training.ppo_runner import _ensure_jax_brax_compat, _load_policy_inference_fn, _resolve_latest_checkpoint

_ensure_jax_brax_compat()

# Tìm checkpoint tối ưu nhất
run_dirs = sorted(glob.glob("results/flat_optimized/g1_m2_larger_mlp_*"))
assert run_dirs, "Chưa tìm thấy thư mục checkpoint!"
ckpt_dir = os.path.join(run_dirs[-1], "checkpoints")
ckpt = _resolve_latest_checkpoint(ckpt_dir)
print(f"[+] Sử dụng checkpoint: {ckpt}")

cfg = load_training_config("configs/larger_mlp.yaml")
env, _, _ = build_g1_environment(cfg)

obs_size = env.observation_size
action_size = int(env.action_size)
has_dict_obs = isinstance(obs_size, dict)

spec = build_model_spec_from_config(cfg, obs_size, action_size)
network_factory = make_model_network_factory(spec, has_dict_obs=has_dict_obs)
policy_fn = _load_policy_inference_fn(
    checkpoint_path=ckpt,
    network_factory=network_factory,
    obs_size=obs_size,
    action_size=action_size,
    deterministic=True,
)

jit_policy = jax.jit(policy_fn)
jit_reset = jax.jit(env.reset)
jit_step = jax.jit(env.step)

# Mô phỏng 600 bước (~12 giây di chuyển liên tục)
key = jax.random.PRNGKey(42)
key, reset_key = jax.random.split(key)
state = jit_reset(reset_key)
trajectory = [state.data]

print("[+] Đang mô phỏng chuyển động bước đi của robot trên mặt phẳng...")
for t in range(600):
    key, act_key = jax.random.split(key)
    action, _ = jit_policy(state.obs, act_key)
    state = jit_step(state, action)
    trajectory.append(state.data)
    if bool(state.done):
        print(f"[i] Episode kết thúc tại bước {t + 1}/600")
        break

# Render khung hình RGB
mj_model = getattr(env, "mj_model", getattr(env, "_mj_model", None))
renderer = mujoco.Renderer(mj_model, height=480, width=640)

cam_id = -1
for c_name in ["track", "side", "back"]:
    try:
        cam_id = mujoco.mj_name2id(mj_model, mujoco.mjtObj.mjOBJ_CAMERA, c_name)
        if cam_id >= 0: break
    except:
        pass
if cam_id == -1 and mj_model.ncam > 0:
    cam_id = 0

mj_data = mujoco.MjData(mj_model)
frames = []

for d in trajectory:
    try:
        mjx.get_data_into(result=mj_data, model=mj_model, data=d)
    except:
        mj_data.qpos[:] = np.asarray(d.qpos)
        mj_data.qvel[:] = np.asarray(d.qvel)
        mujoco.mj_forward(mj_model, mj_data)
    
    if cam_id >= 0:
        renderer.update_scene(mj_data, camera=cam_id)
    else:
        renderer.update_scene(mj_data)
    frames.append(renderer.render())

video_output = "/kaggle/working/demo_flat_walking.mp4"
imageio.mimsave(video_output, frames, fps=30, macro_block_size=None)
print(f"[✓] Đã render xong video demo tại: {video_output}")

# Phát video trực tiếp trong Jupyter Notebook Kaggle
mp4_bytes = open(video_output, "rb").read()
data_url = "data:video/mp4;base64," + b64encode(mp4_bytes).decode()
display(HTML(f"""
<div align="center">
    <h3>Demo Robot Unitree G1 Đi Mượt Trên Mặt Phẳng (Flat Terrain)</h3>
    <video width="640" height="480" controls autoplay loop>
        <source src="{data_url}" type="video/mp4">
    </video>
</div>
"""))